# Phase 2 — Task 4: SQL Fundamentals (Python version)

## Objective

This notebook builds a small SQLite database from the two cleaned NorthStar
CSVs using hand-written `CREATE TABLE` DDL, then answers every required
business question using SQL only, no pandas in the answer cells. pandas is
used only to display query results readably).

**Input:** `dataset/northstar_clean.csv`, `dataset/northstar_regions.csv`.
**Output:** `northstar_python.db` (kept separate from the DBeaver-built
`northstar.db` so this doesn't overwrite that deliverable).

In [1]:
# since sqlite3 is part of the Python standard library, no install needed.
import sqlite3
import pandas as pd

# Connect to and create, if it doesn't exist, a SEPARATE database file,
# so this never touches the northstar.db built in DBeaver.
conn = sqlite3.connect("northstar_python.db")
cur = conn.cursor()

# Drop tables first so this cell can be re-run cleanly from scratch.
cur.executescript("""
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS regions;
""")

print("Connected to northstar_python.db, ready for schema.")

Connected to northstar_python.db, ready for schema.


### Schema (CREATE TABLE)

Column type choices:
- `region`, `regional_manager`, `country`, `payment_method`, `product_category`
  → `TEXT`. Free-form category labels, never used in arithmetic.
- `order_id`, `customer_id` → `TEXT`. Identifiers, not numeric quantities.
- `order_date`, `launch_date` → `DATE`. SQLite stores this as ISO-8601 text
  internally, but declaring it `DATE` documents intent, and ISO-8601 strings
  sort/compare correctly as plain text (so `order_date > '2024-06-01'` works).
- `quantity` → `INTEGER`. Units ordered are always whole numbers.
- `unit_price`, `total_amount` → `REAL`. Money values with decimal pence.
- `discount_pct` → `INTEGER`. Task 1's EDA showed this only ever takes whole
  values (0, 5, 10, 15, 20).
- `returned` → `TEXT`. A 'Y'/'N' flag; SQLite has no native boolean type.

In [2]:
# Write the DDL by hand rather than letting a CSV import auto-generate it.
cur.executescript("""
CREATE TABLE regions (
    region              TEXT PRIMARY KEY,
    regional_manager    TEXT NOT NULL,
    country              TEXT NOT NULL,
    launch_date          DATE NOT NULL
);

CREATE TABLE orders (
    order_id            TEXT PRIMARY KEY,
    order_date          DATE NOT NULL,
    customer_id         TEXT NOT NULL,
    region               TEXT NOT NULL,
    product_category     TEXT NOT NULL,
    quantity             INTEGER NOT NULL,
    unit_price            REAL NOT NULL,
    discount_pct           INTEGER NOT NULL,
    payment_method           TEXT NOT NULL,
    total_amount              REAL NOT NULL,
    returned                   TEXT NOT NULL,
    FOREIGN KEY (region) REFERENCES regions(region)
);
""")
conn.commit()
print("Tables created.")

Tables created.


In [3]:
# Load each CSV with pandas (purely as a convenient CSV reader here, not
# for analysis), then insert rows into the tables already created above
# with executemany, rather than using df.to_sql (which would let pandas
# invent its own schema).

orders_df = pd.read_csv("../../dataset/northstar_clean.csv")
regions_df = pd.read_csv("../../dataset/northstar_regions.csv")

cur.executemany(
    "INSERT INTO regions VALUES (?, ?, ?, ?)",
    regions_df[["region", "regional_manager", "country", "launch_date"]].itertuples(index=False, name=None)
)

cur.executemany(
    "INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
    orders_df[[
        "order_id", "order_date", "customer_id", "region", "product_category",
        "quantity", "unit_price", "discount_pct", "payment_method",
        "total_amount", "returned",
    ]].itertuples(index=False, name=None)
)
conn.commit()

print("Regions loaded:", cur.execute("SELECT COUNT(*) FROM regions").fetchone()[0])
print("Orders loaded:", cur.execute("SELECT COUNT(*) FROM orders").fetchone()[0])

Regions loaded: 8
Orders loaded: 999


In [4]:
# Small helper: runs a SQL string and returns the result as a DataFrame,
# purely for readable display. The query itself is still pure SQL.
def q(sql):
    return pd.read_sql_query(sql, conn)

## 1. SELECT / WHERE / ORDER BY / LIMIT

In [6]:
# Find the 20 highest-value orders overall.
q("""
SELECT order_id, region, total_amount
FROM orders
ORDER BY total_amount DESC
LIMIT 20;
""")

,order_id,region,total_amount
0,NS-00887,North,1974.00
1,NS-00094,South,1929.90
2,NS-00388,South,1890.20
3,NS-00200,Wales,1865.00
4,NS-00243,East,1839.90
5,NS-00865,Scotland,1804.52
6,NS-00311,East,1774.70
7,NS-00908,North,1764.80
8,NS-00249,London,1761.30
9,NS-00514,Wales,1748.79


**Read:** the highest single order is £1,974.00 (North), and the top 20 span roughly £1,700-£1,974.

In [7]:
# Filter for a specific region, date range, and discount threshold at once.
q("""
SELECT order_id, order_date, region, discount_pct, total_amount
FROM orders
WHERE region = 'London'
  AND order_date > '2024-06-01'
  AND discount_pct > 10
ORDER BY order_date;
""")

,order_id,order_date,region,discount_pct,total_amount
0,NS-00786,2024-08-02,London,15,1105.03
1,NS-00981,2024-08-03,London,15,114.78
2,NS-00782,2024-08-20,London,20,268.08
3,NS-00123,2024-08-26,London,15,402.16
4,NS-00714,2024-10-05,London,15,900.59
5,NS-00524,2024-10-28,London,15,658.16
6,NS-00492,2024-11-12,London,15,16.93
7,NS-00409,2024-12-07,London,15,521.12
8,NS-00355,2024-12-19,London,15,680.75


**Read:** 9 London orders in the second half of 2024 exceeded a 10% discount, all landing at exactly 15% or 20%.

## 2. Aggregation

In [8]:
# Sum every order's value into one total.
q("SELECT ROUND(SUM(total_amount), 2) AS total_revenue FROM orders;")

,total_revenue
0,513619.05


**Read:** total revenue across all 999 cleaned orders is £513,619.05.

In [9]:
# Count how many distinct customers placed orders.
q("SELECT COUNT(DISTINCT customer_id) AS unique_customers FROM orders;")

,unique_customers
0,290


**Read:** 290 unique customers, so the average customer placed roughly 3-4 orders over the year.

In [10]:
# Get the smallest, largest, and average basket size in one pass.
q("""
SELECT MIN(quantity) AS min_basket, MAX(quantity) AS max_basket,
       ROUND(AVG(quantity), 2) AS avg_basket
FROM orders;
""")

,min_basket,max_basket,avg_basket
0,1,10,5.51


**Read:** basket size ranges 1-10 units, averaging 5.51. The max of 10 (not 500) confirms Task 2's outlier correction carried through correctly.

## 3. GROUP BY / HAVING

In [11]:
# Total revenue for each region and rank them.
q("""
SELECT region, ROUND(SUM(total_amount), 2) AS revenue
FROM orders
GROUP BY region
ORDER BY revenue DESC;
""")

,region,revenue
0,North,89670.44
1,South,84435.95
2,London,73878.89
3,East,65605.20
4,Midlands,57027.34
5,West,56225.58
6,Scotland,51617.42
7,Wales,35158.23


**Read:** North leads at £89,670.44, Wales trails at £35,158.23, matching Task 3's pandas groupby exactly.

In [12]:
# Keep only categories whose total revenue clears £10,000.
q("""
SELECT product_category, ROUND(SUM(total_amount), 2) AS revenue
FROM orders
GROUP BY product_category
HAVING SUM(total_amount) > 10000
ORDER BY revenue DESC;
""")

,product_category,revenue
0,Sports,83058.95
1,Electronics,82798.89
2,Home & Garden,80202.25
3,Beauty,73339.00
4,Food & Drink,71544.19
5,Clothing,68815.93
6,Toys,53859.84


**Read:** all 7 categories clear £10,000; Sports is highest at £83,058.95, Toys lowest at £53,859.84.

In [13]:
# Keep only regions with more than 100 orders.
q("""
SELECT region, COUNT(*) AS order_count
FROM orders
GROUP BY region
HAVING COUNT(*) > 100
ORDER BY order_count DESC;
""")

,region,order_count
0,North,169
1,South,163
2,London,143
3,West,125
4,East,122
5,Midlands,111


**Read:** 6 of 8 regions clear 100 orders; Scotland and Wales fall below the threshold.

## 4. JOIN

In [14]:
# Attach each order's regional manager by matching on region.
q("""
SELECT o.order_id, o.region, r.regional_manager, o.total_amount
FROM orders AS o
INNER JOIN regions AS r ON o.region = r.region
LIMIT 5;
""")

,order_id,region,regional_manager,total_amount
0,NS-00522,West,Daniel Hughes,1217.37
1,NS-00740,London,Olivia Clarke,84.89
2,NS-00824,Scotland,Fiona MacLeod,14.67
3,NS-00663,London,Olivia Clarke,1603.98
4,NS-00412,Midlands,Mohammed Iqbal,142.65


In [15]:
# Compare row counts before and after the join to prove no orders were silently dropped.
orders_count = cur.execute("SELECT COUNT(*) FROM orders;").fetchone()[0]
inner_count = cur.execute("""
    SELECT COUNT(*) FROM orders AS o
    INNER JOIN regions AS r ON o.region = r.region;
""").fetchone()[0]

print(f"orders row count: {orders_count}")
print(f"inner join row count: {inner_count}")
print(f"match: {orders_count == inner_count}")

orders row count: 999
inner join row count: 999
match: True


**Read:** both counts are 999 -- the inner join matched every order, proving Task 2's region-casing fix worked.

In [16]:
# Delete one region temporarily to show how a left join handles unmatched rows, then undo the deletion.
cur.execute("BEGIN")
cur.execute("DELETE FROM regions WHERE region = 'Wales';")

result = q("""
SELECT o.order_id, o.region, r.regional_manager
FROM orders AS o
LEFT JOIN regions AS r ON o.region = r.region
WHERE o.region = 'Wales'
LIMIT 5;
""")
print(result)

null_count = cur.execute("""
    SELECT COUNT(*) FROM orders AS o
    LEFT JOIN regions AS r ON o.region = r.region
    WHERE r.regional_manager IS NULL;
""").fetchone()[0]
print(f"\nOrders with NULL regional_manager after deleting Wales: {null_count}")

conn.rollback()
print("Rolled back -- Wales row restored.")

   order_id region regional_manager
0  NS-00681  Wales             None
1  NS-00514  Wales             None
2  NS-00211  Wales             None
3  NS-00060  Wales             None
4  NS-00694  Wales             None

Orders with NULL regional_manager after deleting Wales: 71
Rolled back -- Wales row restored.


**Read:** every Wales order shows a NULL `regional_manager` after the delete, and the NULL count (71) exactly matches the Wales order count -- confirming a left join keeps unmatched rows instead of silently dropping them like an inner join would.

## 5. Subqueries

In [17]:
# Use an inner query to total revenue per region, then let the outer query pick the single highest one.
q("""
SELECT region, total_revenue
FROM (
    SELECT region, SUM(total_amount) AS total_revenue
    FROM orders
    GROUP BY region
) AS region_totals
ORDER BY total_revenue DESC
LIMIT 1;
""")

,region,total_revenue
0,North,89670.44


**Read:** North is the single best region by revenue, £89,670.44.

In [18]:
# Compare every order's value against the overall average, computed by a subquery.
q("""
SELECT order_id, region, total_amount
FROM orders
WHERE total_amount > (SELECT AVG(total_amount) FROM orders)
ORDER BY total_amount DESC
LIMIT 10;
""")

,order_id,region,total_amount
0,NS-00887,North,1974.00
1,NS-00094,South,1929.90
2,NS-00388,South,1890.20
3,NS-00200,Wales,1865.00
4,NS-00243,East,1839.90
5,NS-00865,Scotland,1804.52
6,NS-00311,East,1774.70
7,NS-00908,North,1764.80
8,NS-00249,London,1761.30
9,NS-00514,Wales,1748.79


In [19]:
# Count exactly how many orders clear that average.
above_avg = cur.execute("""
    SELECT COUNT(*) FROM orders
    WHERE total_amount > (SELECT AVG(total_amount) FROM orders);
""").fetchone()[0]
print(f"Total orders above average: {above_avg}")

Total orders above average: 408


In [20]:
# Save everything to disk and close the connection cleanly.
conn.commit()
conn.close()
print("Saved and closed northstar_python.db.")

Saved and closed northstar_python.db.
